# 🎨 SDXL Vintage Illustration Notebook — PRO — версия 20261103080426

Stable Diffusion XL / Turbo / SD 1.5 для иллюстраций «тушь и акварель». Ноутбук состоит из полей и кнопок: весь код лежит в проекте (`scripts/`).

Версия в названии — версия этого ноутбука. Ячейка 🚀 Старт печатает версию из репозитория: если числа разные, открыта старая копия, открой свежую.

### Порядок
1. **🚀 Старт**: первой и один раз за сеанс (репозиторий берётся по умолчанию из поля REPO_NAME; остальные ячейки без Старта подскажут, что запустить) (репозиторий, Drive, кеш моделей, зависимости, конфиг, загрузчики).
2. **🎨 Стиль и проект**: выбери проект и стиль (необязательно). Галерея **🌐 Gradio** запускается после неё и показывает ту же папку сохранения.
3. Режим: **Text2Img**, **Img2Img**, **ControlNet**, **Inpainting**, **Upscale** или **🦁 USA Safari — Генератор**.
Для Img2Img, ControlNet и Inpainting сначала запусти ячейку Upload выше нужного режима. Под ячейкой режима остаётся только миниатюра последнего кадра со ссылкой «🔎 Открыть в галерее» (ссылка работает, когда запущена ячейка 🌐 Gradio). Вся история на отдельной странице 🌐 Gradio (ссылка в выводе ячейки): миниатюры, под выбранной картинкой seed и промпт с кнопкой копирования и скачивание оригинала.


## ✅ Советы

- **seed**: положительное число даёт повторяемый результат, 0 — случайный.
- **Img2Img**: меньше `strength` — ближе к оригиналу, больше — сильнее стиль.
- **ControlNet (Canny)** держит контуры; для другого результата меняй пороги `low_thr` / `high_thr`.
- Если не хватает видеопамяти, запусти в ячейке `free_memory()` и повтори нужный режим.
- Модели скачиваются в каждом новом сеансе Colab (кеш на локальном диске).


In [ ]:
# @title 🚀 Старт: репозиторий, Drive, кеш моделей, зависимости, конфиг (один раз за сеанс, запускать первой)
# Кеш моделей на локальном диске Colab: на Google Drive он ломается (символьные ссылки).
PROGRAM_VERSION = "SD15"        #@param ["SDXL", "SDXL_TURBO", "SD15"]
MODE            = "GPU_OPTIMAL" #@param ["GPU_OPTIMAL", "GPU_BASIC", "CPU"]
OUTPUT_DIR      = "/content/outputs"   #@param {type:"string"}
HF_CACHE_DIR    = "/content/hf_cache"  #@param {type:"string"}
DRIVE_MOUNT     = "/content/drive"     #@param {type:"string"}
REPO_NAME = "sd-colab-gallery"  #@param {type:"string"}
import os, sys
repo_dir = f"/content/{REPO_NAME}"
if not os.path.exists(repo_dir):
    !git clone https://github.com/tekswirl25/{REPO_NAME}.git {repo_dir}
sys.path.insert(0, repo_dir); os.chdir(repo_dir)
from scripts.notebook_setup import setup_all
globals().update(setup_all(repo_dir, PROGRAM_VERSION, MODE, OUTPUT_DIR, DRIVE_MOUNT, HF_CACHE_DIR))

In [ ]:
# @title 🔍 1.1 Тест MODE
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
from scripts.config import init_config

# CPU режим
config_cpu, *_ = init_config(model_variant="SDXL", mode="CPU")
print("CPU test:", config_cpu["DEVICE"], config_cpu["DTYPE"])

# GPU базовый
config_gpu_basic, *_ = init_config(model_variant="SDXL", mode="GPU_BASIC")
print("GPU_BASIC test:", config_gpu_basic["DEVICE"], config_gpu_basic["DTYPE"])

# GPU оптимальный
config_gpu_opt, *_ = init_config(model_variant="SDXL", mode="GPU_OPTIMAL")
print("GPU_OPTIMAL test:", config_gpu_opt["DEVICE"], config_gpu_opt["DTYPE"])

In [ ]:
# @title 🎨 Стиль и проект (необязательно)
# Пустой PROJECT: работают только base_style, color_tone и negative. С проектом подставляются стиль, добавка к промпту,
# негатив, шаги/cfg/размер для Text2Img и папка сохранения на Drive (prompts/, см. prompts/README.md).
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
base_style = "illustration"  #@param ["illustration","photoreal","anime"]
color_tone = "vintage"       #@param ["warm","cool","vintage"]
negative   = ""              #@param {type:"string"}
PROJECT    = "a_delicious_piece_of_meat"  #@param ["", "a_delicious_piece_of_meat", "usa_safari"]
STYLE      = ""   #@param {type:"string"}
CHARACTER  = ""   #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
from scripts.style_presets import apply_style
globals().update(apply_style(PROJECT, STYLE, CHARACTER, SAVE_TO_DRIVE, CONFIG, base_style, color_tone, negative))

In [ ]:
# @title 🌐 Start Gradio server (Logs + Gallery)
# Запускай после «Стиль и проект»: галерея показывает папку, куда сохраняют режимы (CONFIG["OUTPUT_DIR"]).
# Вся история на отдельной странице по ссылке из вывода: миниатюры, под выбранной картинкой seed и промпт (копируются) и скачивание.
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
refresh_interval = 5  #@param {type:"integer"}
LOG_LINES = 50        #@param {type:"integer"}
INLINE = False        #@param {type:"boolean"}

from scripts.server_gradio import start_gradio_server
from scripts.notebook_view import show_inline

start_gradio_server(CONFIG["OUTPUT_DIR"], refresh_interval=refresh_interval, LOG_LINES=LOG_LINES)
if INLINE:
    show_inline(7860)   # по желанию: галерея под ячейкой

In [ ]:
# @title 🦁 USA Safari — Генератор
# Весь код в scripts/usa_safari_ui.py. Выбери режим, опиши сцену, нажми GENERATE.
# Результат сохраняется в папку из ячейки «Проект и стиль» (или в OUTPUT_DIR).
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
from scripts.usa_safari_ui import show_generator

show_generator(CONFIG)

In [ ]:
# @title 📁 USA Safari — Drive Reference Setup
# Создаёт папки USA Safari Reference на Google Drive. Запускай один раз после монтирования Drive.
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
from scripts.usa_safari_drive import setup_drive_reference, format_setup

print(format_setup(setup_drive_reference()))

In [ ]:
# @title 🖼 Text2Img
# Шаги, cfg и размер берутся из ячейки «Проект и стиль» (если она запускалась), иначе из настроек варианта.
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
user_prompt = "A thoughtful middle-aged man in a gray coat."  #@param {type:"string"}
seed = 12345   #@param {type:"number"}
n    = 1       #@param {type:"number"}

from scripts.generate import run_text2img

run_text2img(CONFIG, DEFAULTS, user_prompt, seed, n, style_suffix=style_suffix, negative=negative,
             active_preset=globals().get("active_preset"))

In [ ]:
# @title 📤 Upload image (Img2Img)
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
from scripts.notebook_uploads import show_upload

show_upload("img2img")

In [ ]:
# @title 🖼 Img2Img — unified save & metadata (thumbnails, JSON, folders)
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
strength    = 0.6   #@param {type:"number"}
seed        = 12345 #@param {type:"number"}
user_prompt = ""    #@param {type:"string"}
from scripts.generate import run_img2img

run_img2img(CONFIG, DEFAULTS, user_prompt, seed, strength, base_style, color_tone, style_extra, negative)

In [ ]:
# @title 📤 Upload image (ControlNet)
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
from scripts.notebook_uploads import show_upload

show_upload("controlnet")

In [ ]:
# @title 🧪 [TEST] ControlNet pipeline state check
# Run before ControlNet cell to verify txt2img pipe is loaded.
# Remove or skip this cell once ControlNet is stable.
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
from scripts.loaders import _txt2img_pipe, _txt2img_id
print("txt2img pipe loaded:", _txt2img_pipe is not None)
print("txt2img model id  :", _txt2img_id)

In [ ]:
# @title 🧭 ControlNet — unified save & metadata (thumbnails, JSON, folders)
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
low_thr     = 100   #@param {type:"number"}
high_thr    = 200   #@param {type:"number"}
seed        = 12345 #@param {type:"number"}
user_prompt = ""    #@param {type:"string"}
from scripts.generate import run_controlnet

run_controlnet(CONFIG, DEFAULTS, user_prompt, seed, low_thr, high_thr,
               base_style, color_tone, style_extra, negative)

In [ ]:
# @title 📤 Upload image (Inpainting)
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
from scripts.notebook_uploads import show_upload

show_upload("inpaint_image")

In [ ]:
# @title 🎭 Upload mask (Inpainting)
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
from scripts.notebook_uploads import show_upload

show_upload("inpaint_mask")

In [ ]:
# @title 🎭 Inpainting — replace masked area
# Маска: БЕЛОЕ = заменить, ЧЁРНОЕ = оставить. strength: 1.0 = полностью заново, 0.5 = смешать с оригиналом.
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
user_prompt = ""     #@param {type:"string"}
seed        = 12345  #@param {type:"number"}
strength    = 0.99   #@param {type:"number"}
from scripts.generate import run_inpaint

run_inpaint(CONFIG, DEFAULTS, user_prompt, seed, strength, base_style, color_tone, style_extra, negative)

In [ ]:
# @title ⬆️ Upscale x4
assert "CONFIG" in globals(), "Сначала запусти ячейку 🚀 Старт (она одна на весь сеанс)"
in_path = ""        #@param {type:"string"}
seed    = 12345     #@param {type:"number"}
prompt  = ""        #@param {type:"string"}
from scripts.generate import run_upscale

run_upscale(CONFIG, in_path, seed, prompt)